In [1]:
import os


In [2]:
%pwd

'd:\\Text-summerizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Text-summerizer'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir:Path
    data_path:Path
    tokenizer_name:Path

In [9]:
from textsummarizer.constants import *
from textsummarizer.utils.common import read_yaml, create_directories 



In [ ]:
class configurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=CONFIG_FILE_NAME):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation_config

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config

In [12]:
import os
from textsummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [14]:
class DataTranformation:
    def __init__(self,config:DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    def convert_examples_to_features(example_batch):
        input_encodings = tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )

        target_encodings = tokenizer(
            text_target=example_batch['summary'],
            max_length=128,
            truncation=True
        )
        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']

        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)

        dataset_samsum_pt = dataset_samsum.map(
            self.convert_examples_to_features,
            batched=True
        )

        dataset_samsum_pt.save_to_disk(
            os.path.join(self.config.root_dir, "samsum_dataset")
        )

In [18]:
try:
    config = configurationManager()
    data_transformation_config = config.get_data_transformation_config()

    data_transformation = DataTransformation(
        config=data_transformation_config
    )

    data_transformation.convert()

except Exception as e:
    raise e

[2026-09-15 08:11:25,998]:INFO:textsummarizerLogger: yaml file: config\config.yaml loaded successfully


[2026-09-15 08:11:26,011]:INFO:textsummarizerLogger: yaml file: params.yaml loaded successfully


BoxKeyError: "'ConfigBox' object has no attribute 'data_transformation_config'"